In [7]:
import pandas as pd
import geopandas as gpd

In [18]:
data_url= "https://data.wprdc.org/datastore/dump/2c13021f-74a9-4289-a1e5-fe0472c89881"
crashes = pd.read_csv(data_url, low_memory=False)

crashes["DEC_LAT"] = pd.to_numeric(crashes["DEC_LAT"], errors="coerce")
crashes["DEC_LONG"] = pd.to_numeric(crashes["DEC_LONG"], errors="coerce")
crashes = crashes.dropna(subset=["DEC_LAT", "DEC_LONG"])

crashes = crashes[ crashes["DEC_LAT"].between(40.360, 40.520) & crashes["DEC_LONG"].between(-80.100, -79.850)]

print("Crashes inside Allegheny County area:", len(crashes))

crashes["DEC_LAT"] = crashes["DEC_LAT"].round(3)
crashes["DEC_LONG"] = crashes["DEC_LONG"].round(3)

location_counts = crashes.groupby(["DEC_LAT", "DEC_LONG"]).size()
least_crashes = location_counts.sort_values().head(100)

print("\n Locations with the Fewest Crashes:")
print(least_crashes.to_string())

most_crashes = location_counts.sort_values(ascending=False).head(100)

print("\n100 Locations with the Most Crashes:")
print(most_crashes.to_string())

Crashes inside Allegheny County area: 122881

 Locations with the Fewest Crashes:
DEC_LAT  DEC_LONG
40.360   -79.870     1
         -79.868     1
         -79.976     1
         -79.960     1
         -79.956     1
         -80.019     1
         -80.008     1
         -79.995     1
40.414   -80.017     1
         -80.012     1
         -80.005     1
40.487   -80.034     1
         -80.030     1
         -80.011     1
         -80.076     1
40.520   -79.951     1
40.487   -79.921     1
40.414   -80.001     1
         -80.000     1
         -79.998     1
40.487   -79.970     1
         -79.964     1
         -79.962     1
40.414   -80.024     1
         -80.084     1
         -80.078     1
         -79.986     1
         -79.984     1
         -79.983     1
         -79.982     1
         -79.980     1
         -79.979     1
40.362   -80.076     1
         -80.072     1
         -80.063     1
         -80.059     1
         -80.057     1
         -80.095     1
40.454   -79.877     1
   

The Metric
My metric for determining the best neighboor is the number of reported car crashes at each location. I am using the Allegheny County Crash Data dataset, which contains the latitude and longitude coordinates of reported crashes. I  grouped crashes based on their coordinates to identify which locations have the fewest crashes.

The Best Location
After organizing the crash data by latitude and longitude, I ranked the locations based on their total number of crashes. The location with the fewest reported crashes will be considered the best according to my metric. I  took the cordianates of the area witht he least crashes to determine which neighboorhood has the least crashes. **Note:can comapred to the neighboor hood with the most crashes.**

dec long bounareis for least crashes 
-79.851, -80.095
dec lat bounarie for least crashes 
40.520, 40.360

dec long bounareis for most crashes 
-79.908, 
dec lat bounarie for most crashes 
40.479, 40.396


Conclusion 
Based on my analysis of the car crash data, I found that areas around Downtown Pittsburgh and surrounding central neighborhoods had some of the highest numbers of recorded crashes. The coordinates with the most crashes were around 40.44 latitude and -80.00 longitude, with one location having 2,409 crashes. In comparison, many locations with the fewest crashes were farther outside the city center and had only one or two recorded crashes.

This shows that the areas with the most crashes are not necessarily the worst neighborhoods, since busier areas may have more traffic, commuters, and intersections. Personally, I would consider a neighborhood with fewer crashes safer and more desirable, but I would also consider other factors such as accessibility, transportation, and the community. Overall, the data helps identify areas with higher crash concentrations, but it does not determine which neighborhood is the best place to live.


In [12]:
# Download crash dataset
data_url = "https://data.wprdc.org/datastore/dump/2c13021f-74a9-4289-a1e5-fe0472c89881"
crashes = pd.read_csv(data_url, low_memory=False)

# Convert coordinates into numbers
crashes["DEC_LAT"] = pd.to_numeric(crashes["DEC_LAT"], errors="coerce")
crashes["DEC_LONG"] = pd.to_numeric(crashes["DEC_LONG"], errors="coerce")

# Remove crashes with missing coordinates
crashes = crashes.dropna(subset=["DEC_LAT", "DEC_LONG"])

# Keep crashes in Pittsburgh bounadires 
crashes = crashes[crashes["DEC_LAT"].between(40.360, 40.520) & 
    crashes["DEC_LONG"].between(-80.100, -79.850)]
print("Crashes inside Pittsburgh area:", len(crashes))

# Download Pittsburgh neighborhood boundaries
#used AI help here ask TA about that***
neighborhood_url = (
    "https://services1.arcgis.com/YZCmUqbcsUpOKfj7/arcgis/rest/services/"
    "PGHWebNeighborhoods/FeatureServer/0/query"
    "?where=1%3D1&outFields=hood&returnGeometry=true&outSR=4326&f=geojson"
)

neighborhoods = gpd.read_file(neighborhood_url)

# Convert crash coordinates into geographic points
crashes_geo = gpd.GeoDataFrame(crashes,
    geometry=gpd.points_from_xy(crashes["DEC_LONG"],crashes["DEC_LAT"]),
    crs="EPSG:4326"
)


# Make sure both datasets use the same coordinate system
neighborhoods = neighborhoods.to_crs("EPSG:4326")

# Match each crash to its neighborhood
crash_to_neighborhood = gpd.sjoin(
    crashes_geo,
    neighborhoods,
    how="left",
    predicate="within"
)

# Count crashes in each neighborhood
neighborhood_counts = crash_to_neighborhood.groupby("hood").size()

# Sort neighborhoods from most crashes to fewest
neighborhood_counts = neighborhood_counts.sort_values()

# Print all neighborhoods and their crash counts
print("Crashes in Each Neighborhood")
for neighborhood, count in neighborhood_counts.items():
    print(neighborhood, " -> ", count)

Crashes inside Pittsburgh area: 122881
Crashes in Each Neighborhood
Arlington Heights  ->  9
Mt. Oliver  ->  44
East Carnegie  ->  58
Chartiers City  ->  66
New Homestead  ->  78
Glen Hazel  ->  90
Oakwood  ->  100
Fairywood  ->  131
Regent Square  ->  133
Spring Garden  ->  142
St. Clair  ->  163
Windgap  ->  165
California-Kirkbride  ->  168
Bedford Dwellings  ->  175
Upper Hill  ->  219
Arlington  ->  230
Fineview  ->  231
Terrace Village  ->  249
Ridgemont  ->  273
Manchester  ->  286
Allegheny Center  ->  287
Allegheny West  ->  303
Esplen  ->  304
Friendship  ->  354
Upper Lawrenceville  ->  356
Westwood  ->  356
Northview Heights  ->  393
Central Northside  ->  397
Middle Hill  ->  404
Morningside  ->  446
Crafton Heights  ->  448
Spring Hill-City View  ->  479
West Oakland  ->  499
Stanton Heights  ->  505
Allentown  ->  508
Lincoln Place  ->  527
Beltzhoover  ->  545
Bon Air  ->  566
Sheraden  ->  570
Knoxville  ->  589
Homewood North  ->  590
Garfield  ->  591
Polish Hill  ->